# G31-04 Data preparation (Colab)
Group G31-04, Dataset S1. Data Engineer part: cleaning, weekly item table, demand features, data dictionary, cleaning log.

**What this notebook does.** It clones the GitHub repository, copies the 4 raw CSV files from Google Drive into `data_raw/`,
and runs the numbered scripts in `scripts/` (steps 1 to 7). All numbers below come from these scripts. Nothing is typed by hand.

Step 7 makes every chart for the slides and the report.

**How to use** (full guide: `HOW_TO_USE.md` in the repository)
1. In cell 1, `RAW_DRIVE_DIR` is the Google Drive folder with the 4 raw CSVs and `SAVE_DRIVE_DIR` the folder for the results.
   Both can be a folder link or a path. With links, Colab asks you to sign in with Google; use the account that can open both folders.
3. *Runtime > Run all*. A full run takes about 5 to 10 minutes.
4. For the submission: *File > Print > Save as PDF* gives the PDF with outputs (`G31-04_Progress1_DataPrep.pdf`).


In [ ]:
#@title 1. Settings
REPO_URL = "https://github.com/RafiSadad/g31-04-stockout-kpi.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
# A Google Drive folder link, or a path such as /content/drive/MyDrive/G31-04
RAW_DRIVE_DIR = "https://drive.google.com/drive/folders/1L2N2pFDTrI8cf2VQQMiS9PuqRIN0EuIq"  #@param {type:"string"}
SAVE_DRIVE_DIR = "https://drive.google.com/drive/folders/1N7ZhSxAoZrTyzmRgChgoF7F6Fel9pWrm"  #@param {type:"string"}

import os, re
# Only for running outside Colab (testing): these environment variables replace the values above.
REPO_DIR = os.environ.get("NB_LOCAL_REPO", "/content/g31-04-stockout-kpi")
RAW_DRIVE_DIR = os.environ.get("NB_RAW_DIR", RAW_DRIVE_DIR)
SAVE_DRIVE_DIR = os.environ.get("NB_SAVE_DIR", SAVE_DRIVE_DIR)

def folder_id(value):
    """Folder ID from a Drive link (or a bare ID); None for a normal path."""
    m = re.search(r"/folders/([\w-]+)", value)
    if m:
        return m.group(1)
    return value if re.fullmatch(r"[\w-]{20,}", value) else None

RAW_ID, SAVE_ID = folder_id(RAW_DRIVE_DIR), folder_id(SAVE_DRIVE_DIR)
print("raw data :", f"Drive folder id {RAW_ID}" if RAW_ID else RAW_DRIVE_DIR)
print("results  :", f"Drive folder id {SAVE_ID}" if SAVE_ID else SAVE_DRIVE_DIR)

In [ ]:
#@title 2. Connect Google Drive and get the code from GitHub
import subprocess, sys
IN_COLAB = "google.colab" in sys.modules
svc = None
if IN_COLAB:
    if RAW_ID or SAVE_ID:   # folder links: use the Drive API with your Google account (a sign-in window opens)
        from google.colab import auth
        from googleapiclient.discovery import build
        auth.authenticate_user()
        svc = build("drive", "v3")
    if not (RAW_ID and SAVE_ID):   # a path is used: mount Google Drive
        from google.colab import drive
        drive.mount("/content/drive")
    url = REPO_URL
    try:  # private repository: add a GitHub token as a Colab secret named GITHUB_TOKEN (key icon on the left)
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        pass
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q", "origin", BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("repository:", REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--format=commit %h, %cd, %s"], capture_output=True, text=True).stdout)

# ---- small Drive API helpers, used only when a folder link is given ----
def drive_list(fid):
    files, token = [], None
    while True:
        r = svc.files().list(q=f"'{fid}' in parents and trashed=false", pageSize=200, pageToken=token,
                             fields="nextPageToken, files(id, name, size, mimeType)",
                             supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        files += r.get("files", [])
        token = r.get("nextPageToken")
        if not token:
            return files

def drive_download(file_id, dest):
    from googleapiclient.http import MediaIoBaseDownload
    with open(dest, "wb") as fh:
        dl = MediaIoBaseDownload(fh, svc.files().get_media(fileId=file_id, supportsAllDrives=True), chunksize=100 * 1024 * 1024)
        done = False
        while not done:
            _, done = dl.next_chunk()

def drive_subfolder(name, parent_id):
    for f in drive_list(parent_id):
        if f["name"] == name and f["mimeType"] == "application/vnd.google-apps.folder":
            return f["id"]
    body = {"name": name, "parents": [parent_id], "mimeType": "application/vnd.google-apps.folder"}
    return svc.files().create(body=body, fields="id", supportsAllDrives=True).execute()["id"]

def drive_upload(path, parent_id):
    """Upload one file; a file with the same name in that folder is replaced (new version), not duplicated."""
    from googleapiclient.http import MediaFileUpload
    name = os.path.basename(path)
    same = [f["id"] for f in drive_list(parent_id) if f["name"] == name]
    media = MediaFileUpload(path, resumable=True)
    if same:
        svc.files().update(fileId=same[0], media_body=media, supportsAllDrives=True).execute()
    else:
        svc.files().create(body={"name": name, "parents": [parent_id]}, media_body=media, fields="id", supportsAllDrives=True).execute()

In [ ]:
#@title 3. Copy the 4 raw CSV files into data_raw/
import glob, shutil, zipfile
NEEDED = ["category_tree.csv", "events.csv", "item_properties_part1.csv", "item_properties_part2.csv"]
os.makedirs("data_raw", exist_ok=True)
if RAW_ID:   # Drive folder link: download with the Drive API
    remote = {f["name"]: f for f in drive_list(RAW_ID)}
    for name in NEEDED:
        if name in remote and not os.path.exists(f"data_raw/{name}"):
            print("downloading", name, f"({int(remote[name].get('size', 0)) / 1e6:.0f} MB)")
            drive_download(remote[name]["id"], f"data_raw/{name}")
    for name, f in remote.items():   # the files may also be inside a zip in that folder
        if name.endswith(".zip") and any(not os.path.exists(f"data_raw/{n}") for n in NEEDED):
            drive_download(f["id"], f"data_raw/{name}")
    search_dir = "data_raw"
else:
    search_dir = RAW_DRIVE_DIR
for name in NEEDED:
    if os.path.exists(f"data_raw/{name}"):
        continue
    hits = glob.glob(f"{search_dir}/**/{name}", recursive=True)
    if hits:
        shutil.copy(hits[0], f"data_raw/{name}")
        continue
    for z in glob.glob(f"{search_dir}/**/*.zip", recursive=True):
        with zipfile.ZipFile(z) as zf:
            member = next((m for m in zf.namelist() if m.endswith(name)), None)
            if member:
                with zf.open(member) as src, open(f"data_raw/{name}", "wb") as dst:
                    shutil.copyfileobj(src, dst)
                break
missing = [n for n in NEEDED if not os.path.exists(f"data_raw/{n}")]
if missing:
    raise FileNotFoundError(f"Not found in {RAW_DRIVE_DIR}: {missing}. Check RAW_DRIVE_DIR in cell 1.")
for n in NEEDED:
    size = os.path.getsize(f"data_raw/{n}")
    check = ""
    if RAW_ID and n in remote and remote[n].get("size"):
        check = "same size as on Drive" if int(remote[n]["size"]) == size else "SIZE DIFFERS FROM DRIVE, run this cell again"
    print(f"{n:28s} {size:>13,} bytes  {check}")
os.environ["DATA_DIR"] = "data_raw"

In [ ]:
#@title 4. Install libraries and record versions
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
import platform, pandas, numpy, openpyxl
print("python", platform.python_version(), "| pandas", pandas.__version__, "| numpy", numpy.__version__, "| openpyxl", openpyxl.__version__)

def run(script):
    """Run one numbered script, show its output, stop the notebook if it fails."""
    print(f"===== {script} =====")
    r = subprocess.run([sys.executable, f"scripts/{script}.py"], capture_output=True, text=True)
    print(r.stdout[-6000:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
        raise RuntimeError(f"{script} failed")

In [ ]:
#@title Step 1: read the raw files, keep 2 properties, remove duplicate events
run("01_extract_raw")

In [ ]:
#@title Step 2: check the 18 weekly snapshot dates (prints only)
run("02_explore_snapshots")

In [ ]:
#@title Step 3: weekly item table, label, category fixes, KPI tables, Reconcile
run("03_build_kpis")

In [ ]:
#@title Step 4: demand features
run("04_demand_features")

In [ ]:
#@title Step 5: data dictionary and cleaning log
run("05_docs")

In [ ]:
#@title Step 6: cleaned CSV for the submission, and comparison with the Progress 1 slides
run("06_export_compare")

In [ ]:
#@title Step 7: all charts for the slides and the report (outputs/charts)
run("07_charts")

## Results

In [ ]:
#@title Cleaning log: rows and columns before and after each step
import pandas as pd
pd.set_option("display.max_colwidth", 120)
log = pd.read_csv("work/cleaning_log.csv")
log[["script", "step", "action", "rows_before", "cols_before", "rows_after", "cols_after", "note"]]

In [ ]:
#@title Reconcile: script result vs the numbers in the proposal
pd.read_excel("outputs/G31-04 KPI Tables.xlsx", sheet_name="Reconcile")

In [ ]:
#@title Comparison with the Progress 1 slides (same = the slide number is reproduced by this code)
pd.read_csv("outputs/compare_progress1.csv")

In [ ]:
#@title Modelling table: shape, target balance, feature summary
mt = pd.read_csv("work/modelling_table_features.csv.gz")
print(f"{len(mt):,} rows x {mt.shape[1]} columns | stock-outs: {int(mt.y_next_out.sum()):,} ({100 * mt.y_next_out.mean():.1f}%)")
mt.describe().T[["count", "mean", "50%", "max"]].round(3)

In [ ]:
#@title Data flow diagram and dataset overview
from IPython.display import Image, display
display(Image("docs/data_flow.png", width=1100))
display(Image("docs/dataset_schema.png", width=1100))

In [ ]:
#@title All charts made by step 7
for f in sorted(glob.glob("outputs/charts/*.png")):
    print(f)
    display(Image(f, width=1000))

In [ ]:
#@title Save the results to Google Drive
RESULTS = ["outputs/G31-04 KPI Tables.xlsx", "outputs/G31-04_Progress1_CleanedData.csv", "outputs/compare_progress1.csv",
           "docs/data_dictionary.md", "docs/cleaning_log.md", "work/cleaning_log.csv"]
charts = sorted(glob.glob("outputs/charts/*.png"))
if SAVE_ID:   # Drive folder link: upload with the Drive API (same name = replaced, not duplicated)
    for f in RESULTS:
        drive_upload(f, SAVE_ID)
        print("uploaded", f)
    charts_id = drive_subfolder("charts", SAVE_ID)
    for f in charts:
        drive_upload(f, charts_id)
    print("uploaded", len(charts), "charts to the 'charts' sub-folder")
    print("folder:", SAVE_DRIVE_DIR)
else:
    os.makedirs(f"{SAVE_DRIVE_DIR}/charts", exist_ok=True)
    for f in RESULTS:
        shutil.copy(f, SAVE_DRIVE_DIR)
        print("saved", f)
    for f in charts:
        shutil.copy(f, f"{SAVE_DRIVE_DIR}/charts")
    print("saved", len(charts), "charts to", f"{SAVE_DRIVE_DIR}/charts")

In [ ]:
#@title Optional: push the updated docs and workbook to GitHub
PUSH_TO_GITHUB = False  #@param {type:"boolean"}
GIT_NAME = ""  #@param {type:"string"}
GIT_EMAIL = ""  #@param {type:"string"}
if PUSH_TO_GITHUB:
    assert GIT_NAME and GIT_EMAIL, "Fill GIT_NAME and GIT_EMAIL first."
    g = lambda *a: subprocess.run(["git", *a], check=True)
    g("config", "user.name", GIT_NAME); g("config", "user.email", GIT_EMAIL)
    g("add", "docs/data_dictionary.md", "docs/cleaning_log.md", "outputs/G31-04 KPI Tables.xlsx", "outputs/charts")
    g("commit", "-m", "Regenerate docs, KPI workbook and charts from a Colab run")
    g("push", "origin", BRANCH)
else:
    print("Skipped. Set PUSH_TO_GITHUB to True (and a GITHUB_TOKEN Colab secret with write access) to push.")